# OpenGadda → tables for OpenRefine

This notebook turns the OpenGadda JSON files into flat CSV tables you can load into OpenRefine.
The JSON files are only **read**, never changed. Run all cells again whenever the data changes.

It writes into `lod/openrefine/tables/`:

| File | One row per | Use in OpenRefine |
|---|---|---|
| `people.csv` | person in `people_birth_death.json` | check Wikidata/VIAF IDs and dates |
| `names_in_items.csv` | person per item in `items.json` | find misspellings and missing people |
| `places.csv` | distinct place | cluster variants, strip addresses |
| `periodicals.csv` | distinct journal / publisher | cluster variants |

## 1. Setup
Finds the repo folder (the one containing `code_updated/`), wherever the notebook is opened from.

In [1]:
import csv
import json
import re
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "code_updated").is_dir())
OUT = ROOT / "lod" / "openrefine" / "tables"
OUT.mkdir(parents=True, exist_ok=True)

ITEMS = ROOT / "code_updated" / "items.json"
AUTHORITY = ROOT / "code_updated" / "people_authority.json"
MITTENTI = ROOT / "code_updated" / "mittenti_enriched.json"
LIFESPANS = ROOT / "storyline" / "people_birth_death.json"
print("Repo:", ROOT)

Repo: /Users/martinapensalfini/Documents/opengadda


## 2. Name matching

- `ALIASES`: known misspellings in `items.json` → the name used in `people_birth_death.json`. Add new ones here as you find them.
- `plain()`: `"Bo, Carlo"` → `"Bo Carlo"`, the form used in the lifespans file.
- `key()`: a loose key that ignores accents, punctuation, capitals and word order, so `"Lémaire, Gérard-Georges"` finds `"Lemaire Gérard-Georges"`.

In [2]:
ALIASES = {
    "Carrocci Alberto": "Carocci Alberto",
    "Amicucchi Ermanno": "Amicucci Ermanno",
    "Ferreti Gian Carlo": "Ferretti Gian Carlo",
    "Pieraccini Cecchi Leonetta": "Cecchi Pieraccini Leonetta",
    "Frateili Armando": "Fratelli Armando",
}


def key(name):
    s = unicodedata.normalize("NFKD", name or "")
    s = "".join(c for c in s if not unicodedata.combining(c)).lower()
    return " ".join(sorted(re.findall(r"[a-z0-9]+", s)))


def plain(name):
    return re.sub(r"\s+", " ", (name or "").replace(",", " ")).strip().rstrip(".")


def write(name, header, rows):
    with open(OUT / name, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)
    print(f"{name}: {len(rows)} rows")

## 3. Load the data

In [3]:
items = json.load(open(ITEMS, encoding="utf-8"))
authority = json.load(open(AUTHORITY, encoding="utf-8"))
mittenti = {m["name"]: m for m in json.load(open(MITTENTI, encoding="utf-8"))}
lifespans = {k: v for k, v in json.load(open(LIFESPANS, encoding="utf-8")).items() if isinstance(v, dict)}

life_by_key = {key(n): n for n in lifespans}
auth_by_key = {key(n): n for n in authority}


def match(raw):
    """Return (name in lifespans file, how it was matched)."""
    p = plain(raw)
    if p in lifespans:
        return p, "exact"
    if p in ALIASES:
        return ALIASES[p], "alias"
    k = key(raw)
    if k in life_by_key:
        return life_by_key[k], "loose"
    return "", "none"


print(len(items), "items ·", len(lifespans), "people with lifespans ·", len(authority), "authority entries")

10291 items · 1330 people with lifespans · 2009 authority entries


## 4. `names_in_items.csv`

The `match` column says how each name was found:
**exact**, **alias** (from the list above), **loose** (only after ignoring accents/punctuation/order: a spelling difference to fix in one of the files) or **none** (not in the lifespans file).

In [4]:
rows, seen = [], defaultdict(Counter)
for it in items:
    for person in it.get("people") or []:
        raw = person.get("name", "")
        matched, how = match(raw)
        rows.append([it["id"], it["category"], it.get("year", ""), person.get("role", ""),
                     raw, plain(raw), matched, how])
        seen[matched or plain(raw)][it["category"]] += 1

write("names_in_items.csv",
      ["item_id", "category", "year", "role", "name_as_written", "name_plain",
       "name_in_lifespans", "match"], rows)

print(Counter(r[7] for r in rows))

names_in_items.csv: 10593 rows
Counter({'exact': 7403, 'none': 3167, 'loose': 22, 'alias': 1})


### Loose matches: spelling differences between `items.json` and the lifespans file

In [5]:
for (written, life), n in Counter((r[4], r[6]) for r in rows if r[7] in ("loose", "alias")).most_common():
    print(f"{n:>3} × {written!r:40} → {life!r}")

  6 × 'Dell’Arco, Mario'                       → "Dell'Arco Mario"
  4 × 'Lémaire, Gérard-Georges'                → 'Lemaire Gérard-Georges'
  1 × 'Lehr Luigia'                            → 'Lehr luigia'
  1 × 'GAdda Carlo Emilio'                     → 'Gadda Carlo Emilio'
  1 × 'Ferreti Gian Carlo'                     → 'Ferretti Gian Carlo'
  1 × 'Lasky, Melvin J.'                       → 'Lasky Melvin J.'
  1 × 'Painter, George D.'                     → 'Painter George D.'
  1 × 'Lucente, Gregory L.'                    → 'Lucente Gregory L.'
  1 × 'Clelia Martignoni'                      → 'Martignoni Clelia'
  1 × 'Gianmarco Gaspari'                      → 'Gaspari Gianmarco'
  1 × 'Franco Gavazzeni'                       → 'Gavazzeni Franco'
  1 × 'Maria Antonietta Terzoli'               → 'Terzoli Maria Antonietta'
  1 × 'Guido Lucchini'                         → 'Lucchini Guido'
  1 × 'Bárberi Squarotti, Giorgio'             → 'Bàrberi Squarotti Giorgio'
  1 × 'Gabici, Franco'  

### Names not in the lifespans file, by category

In [6]:
missing = Counter((r[1], r[5]) for r in rows if r[7] == "none")
print(Counter(cat for cat, _ in missing), "distinct names")
for (cat, name), n in missing.most_common(15):
    print(f"{n:>3} × {cat:24} {name}")

Counter({'library': 754, 'secondary bibliography': 549, 'letter': 429, 'translation': 41, 'primary bibliography': 9}) distinct names
 45 × letter                   Neri Pozza editore
 43 × letter                   Calissano Luigi
 37 × secondary bibliography   Pedriali Federica G
 30 × secondary bibliography   Ungarelli Giulio
 28 × letter                   Ammonia Casale
 25 × secondary bibliography   Pinotti Giorgio
 22 × secondary bibliography   Stellardi Giuseppe
 19 × library                  Alighieri Dante
 19 × library                  Manzoni Alessandro
 19 × letter                   Marchetti Domenico
 19 × library                  Shakespeare William
 18 × secondary bibliography   Dombroski Robert S
 17 × library                  Cicerone Marco Tullio
 16 × library                  D’Annunzio Gabriele
 16 × secondary bibliography   Rinaldi Rinaldo


## 5. `people.csv`

One row per person in the lifespans file, with:
`death_status` (**dead** = year given, **alive** = `"morte": null`, **unknown** = no `morte` key), the Wikidata/VIAF IDs from `people_authority.json` (VIAF falls back to `mittenti_enriched.json`), and how many items the person has, by category.

In [7]:
rows = []
for name, v in lifespans.items():
    if "morte" not in v:
        status = "unknown"
    elif v["morte"] is None:
        status = "alive"
    else:
        status = "dead"
    auth = authority.get(auth_by_key.get(key(name), ""), {})
    mit = mittenti.get(name, {})
    cats = seen.get(name, Counter())
    rows.append([
        name, v.get("nascita", ""), v.get("morte", "") or "", status,
        v.get("categoria", ""), v.get("reparto", ""),
        auth.get("wikidata_id") or "", auth.get("wikidata_label") or "",
        auth.get("wikidata_description") or "",
        auth.get("viaf_id") or mit.get("viaf_id") or "",
        sum(cats.values()), "; ".join(f"{c} {n}" for c, n in cats.most_common()),
    ])

write("people.csv",
      ["name", "birth", "death", "death_status", "categoria", "reparto",
       "wikidata_id", "wikidata_label", "wikidata_description", "viaf_id",
       "items", "items_by_category"], rows)

print(Counter(r[3] for r in rows))
print("with Wikidata ID:", sum(1 for r in rows if r[6]), "· with VIAF ID:", sum(1 for r in rows if r[9]),
      "· with no items:", sum(1 for r in rows if r[10] == 0))

people.csv: 1330 rows
Counter({'dead': 1003, 'alive': 323, 'unknown': 4})
with Wikidata ID: 1268 · with VIAF ID: 1197 · with no items: 28


## 6. `places.csv` and `periodicals.csv`

In [8]:
places = Counter()
for it in items:
    for field in ("luogo_di_invio", "luogo_destinatario", "luogo_di_pubblicazione"):
        if it.get(field):
            places[(field, it[field].strip())] += 1
write("places.csv", ["field", "value", "count"],
      [[f, v, n] for (f, v), n in places.most_common()])

venues = Counter()
for it in items:
    for field in ("rivista", "casa_editrice", "editore"):
        if it.get(field):
            venues[(field, it["category"], it[field].strip())] += 1
write("periodicals.csv", ["field", "category", "value", "count"],
      [[f, c, v, n] for (f, c, v), n in venues.most_common()])

places.csv: 458 rows
periodicals.csv: 1953 rows


## Next: in OpenRefine

1. *Create project* → choose `names_in_items.csv`.
2. Text facet on `match` → pick **none**.
3. On `name_plain`: *Edit cells → Cluster and edit* to catch variants of the same name.
4. On the remaining names: *Reconcile → Start reconciling → Wikidata*, type *human*.
5. Export your decisions as a correction table (e.g. `name_corrections.csv`) and keep the operation history (*Undo/Redo → Extract*).